# Notebook 3:
### Hourly Workload and Staffing Coverage Model

**Purpose**

Transforms prescription activity into hourly workload, estimates required staffing, expands weekly schedules into hourly coverage, and computes staffing gaps.

#### Output
- `projected_staffing_by_hour.csv`

**Key responsibilitiy**
- Provides the operational efficiency layer used to evaluate staffing alignment against prescription workload in Power BI.

In [1]:
# --- Setup & paths ---
from pathlib import Path
import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD

DATA_GEN = REPO_ROOT / "data" / "generated"
DATA_REAL = REPO_ROOT / "data" / "real"
BI_OUT   = REPO_ROOT / "bi_outputs"
BI_OUT.mkdir(parents=True, exist_ok=True)

def first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

P_RX = first_existing([
    DATA_GEN / "prescriptions.csv",
    REPO_ROOT / "prescriptions.csv",
])

P_STAFF = first_existing([
    DATA_GEN / "staffschedule.csv",
    REPO_ROOT / "staffschedule.csv",
])

if P_RX is None:
    raise FileNotFoundError("prescriptions.csv not found. Expected at data/generated/prescriptions.csv")
if P_STAFF is None:
    raise FileNotFoundError("staffschedule.csv not found. Expected at data/generated/staffschedule.csv")

print("Repo root:", REPO_ROOT)
print("Using prescriptions:", P_RX)
print("Using staffschedule:", P_STAFF)


Repo root: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization
Using prescriptions: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/data/generated/prescriptions.csv
Using staffschedule: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/data/generated/staffschedule.csv


In [2]:
# --- 1) Load prescriptions and build hourly workload ---
rx = pd.read_csv(P_RX, low_memory=False)

date_col = "FillDate" if "FillDate" in rx.columns else ("RefillDate" if "RefillDate" in rx.columns else ("Date" if "Date" in rx.columns else None))
if date_col is None:
    raise ValueError("prescriptions.csv must include FillDate or RefillDate or Date")

ts = pd.to_datetime(rx[date_col], errors="coerce")
rx = rx.loc[ts.notna()].copy()
rx["Timestamp"] = ts.loc[ts.notna()].values
rx["Date"] = pd.to_datetime(rx["Timestamp"]).dt.normalize()

# If timestamps have no hours (all midnight), synthesize hours with a reasonable distribution
hours = pd.to_datetime(rx["Timestamp"]).dt.hour
OPEN_HOUR, CLOSE_HOUR = 8, 20

if hours.nunique() == 1 and int(hours.iloc[0]) == 0:
    hour_bins = np.arange(OPEN_HOUR, CLOSE_HOUR + 1)
    weights = np.array([0.6, 0.8, 1.0, 1.2, 1.3, 1.25, 1.15, 1.05, 0.95, 0.85, 0.75, 0.65, 0.55])
    weights = weights / weights.sum()
    rx["Hour"] = np.random.default_rng(42).choice(hour_bins, size=len(rx), p=weights)
else:
    rx["Hour"] = hours.astype(int)

# RxCount per hour (count of prescriptions)
hourly_req = (
    rx.groupby(["Date","Hour"], as_index=False)
      .size()
      .rename(columns={"size":"RxCount"})
)

print("Rx date range:", hourly_req["Date"].min().date(), "to", hourly_req["Date"].max().date())
hourly_req.head()

Rx date range: 2023-01-01 to 2024-12-30


,Date,Hour,RxCount
0,2023-01-01,9,1
1,2023-01-01,10,2
2,2023-01-01,11,1
3,2023-01-01,12,4
4,2023-01-01,13,2


In [3]:
# --- 2) Convert weekly staffschedule -> daily hourly scheduled staff ---
staff = pd.read_csv(P_STAFF, low_memory=False)

# Standardize week start date
if "WeekStartDate" not in staff.columns:
    raise ValueError("staffschedule.csv must include WeekStartDate")
staff["WeekStartDate"] = pd.to_datetime(staff["WeekStartDate"], errors="coerce").dt.normalize()

if "Shift" not in staff.columns or "HoursWorked" not in staff.columns:
    raise ValueError("staffschedule.csv must include Shift and HoursWorked")

staff["Shift"] = staff["Shift"].astype(str).str.upper().str.strip()
staff["HoursWorked"] = pd.to_numeric(staff["HoursWorked"], errors="coerce").fillna(0.0)

# Build a full Date x Hour grid from Rx workload
all_days = pd.date_range(hourly_req["Date"].min(), hourly_req["Date"].max(), freq="D")
all_hours = np.arange(OPEN_HOUR, CLOSE_HOUR + 1)

grid = (
    pd.MultiIndex.from_product([all_days, all_hours], names=["Date","Hour"])
      .to_frame(index=False)
)

# Map each Date -> WeekStartDate (Monday)
grid["WeekStartDate"] = (grid["Date"] - pd.to_timedelta(grid["Date"].dt.weekday, unit="D")).dt.normalize()

# Aggregate schedule hours by week + shift
weekly_shift_hours = (
    staff.groupby(["WeekStartDate","Shift"], as_index=False)
         .agg(ShiftHours=("HoursWorked","sum"))
)

# Assumptions to spread weekly hours into hourly coverage:
# - 5 workdays/week (Mon-Fri)
# - AM covers 8-13 (6 hours) and PM covers 14-20 (7 hours)
AM_HOURS = list(range(8, 14))
PM_HOURS = list(range(14, 21))

HOURS_PER_DAY_AM = len(AM_HOURS)
HOURS_PER_DAY_PM = len(PM_HOURS)
WORKDAYS_PER_WEEK = 5

def weekly_to_hourly_staff(week_start, shift, shift_hours):
    """Return scheduled staff per hour for each workday in the week for one shift."""
    if shift_hours <= 0:
        return {}
    if shift == "AM":
        hours = AM_HOURS
        hrs_per_day = HOURS_PER_DAY_AM
    elif shift == "PM":
        hours = PM_HOURS
        hrs_per_day = HOURS_PER_DAY_PM
    else:
        return {}

    # Total shift hours per day (spread evenly across 5 weekdays)
    hours_per_day = shift_hours / WORKDAYS_PER_WEEK
    # Convert to "staff present per hour" for those shift hours
    staff_per_hour = hours_per_day / hrs_per_day
    return {h: staff_per_hour for h in hours}

# Build a lookup: (WeekStartDate) -> {hour -> scheduled staff}
week_hour_map = {}

for ws, sub in weekly_shift_hours.groupby("WeekStartDate"):
    hour_map = {}
    for _, r in sub.iterrows():
        add = weekly_to_hourly_staff(ws, r["Shift"], float(r["ShiftHours"]))
        for h, v in add.items():
            hour_map[h] = hour_map.get(h, 0.0) + float(v)
    week_hour_map[pd.Timestamp(ws).normalize()] = hour_map

def scheduled_for_row(row):
    ws = row["WeekStartDate"]
    h  = int(row["Hour"])
    m = week_hour_map.get(ws, None)
    if m is None:
        return np.nan
    return float(m.get(h, 0.0))

grid["ScheduledStaff"] = grid.apply(scheduled_for_row, axis=1)

# Headcount is a readable proxy for dashboards
grid["ScheduledHeadcount"] = np.where(
    grid["ScheduledStaff"].notna(),
    np.clip(np.rint(grid["ScheduledStaff"]), 0, None).astype(int),
    np.nan
)

grid["ScheduleSource"] = np.where(grid["ScheduledStaff"].notna(), "Staff Schedule", "Missing")

sched = grid[["Date","Hour","ScheduledStaff","ScheduledHeadcount","ScheduleSource"]].copy()
print("sched rows:", len(sched))
print(sched["ScheduleSource"].value_counts())
sched.head(12)

sched rows: 9490
ScheduleSource
Staff Schedule    9490
Name: count, dtype: int64


,Date,Hour,ScheduledStaff,ScheduledHeadcount,ScheduleSource
0,2023-01-01,8,6.192667,6.0,Staff Schedule
1,2023-01-01,9,6.192667,6.0,Staff Schedule
2,2023-01-01,10,6.192667,6.0,Staff Schedule
3,2023-01-01,11,6.192667,6.0,Staff Schedule
4,2023-01-01,12,6.192667,6.0,Staff Schedule
5,2023-01-01,13,6.192667,6.0,Staff Schedule
6,2023-01-01,14,4.848857,5.0,Staff Schedule
7,2023-01-01,15,4.848857,5.0,Staff Schedule
8,2023-01-01,16,4.848857,5.0,Staff Schedule
9,2023-01-01,17,4.848857,5.0,Staff Schedule


In [4]:
# --- 3) Required staff + Over/Under gap ---
RX_PER_STAFF_HOUR = 35
MIN_STAFF = 2.0

# Smooth RxCount slightly
hourly_req2 = hourly_req.copy()
hourly_req2 = hourly_req2.sort_values(["Date","Hour"]).reset_index(drop=True)

# Rolling mean within each day
hourly_req2["RxCount_Smoothed"] = (
    hourly_req2.groupby("Date")["RxCount"]
              .transform(lambda s: s.rolling(window=3, min_periods=1, center=True).mean())
)

hourly_req2["RequiredStaff"] = np.maximum(MIN_STAFF, hourly_req2["RxCount_Smoothed"] / RX_PER_STAFF_HOUR)

# Merge schedule into workload
merged = hourly_req2.merge(sched, on=["Date","Hour"], how="left")

# OverUnder: Scheduled minus Required
merged["OverUnder"] = np.where(
    merged["ScheduleSource"].eq("Staff Schedule") & merged["ScheduledStaff"].notna(),
    (merged["ScheduledStaff"] - merged["RequiredStaff"]),
    0.0
)

# Flags
merged["OverUnderFlag"] = np.select(
    [
        merged["ScheduleSource"].ne("Staff Schedule"),
        merged["OverUnder"] < -1e-9,
        merged["OverUnder"] >  1e-9,
    ],
    [
        "NoScheduleData",
        "Understaffed",
        "Overstaffed",
    ],
    default="Balanced"
)

merged["ScheduledStaff"] = pd.to_numeric(merged["ScheduledStaff"], errors="coerce")
merged["ScheduledHeadcount"] = pd.to_numeric(merged["ScheduledHeadcount"], errors="coerce")
merged["OverUnder"] = pd.to_numeric(merged["OverUnder"], errors="coerce").fillna(0.0)

out = merged[[
    "Date","Hour","RxCount","RxCount_Smoothed",
    "RequiredStaff","ScheduledStaff","ScheduledHeadcount",
    "ScheduleSource","OverUnder","OverUnderFlag"
]].copy()

OUT_PATH = BI_OUT / "projected_staffing_by_hour.csv"
out.to_csv(OUT_PATH, index=False)

print("Wrote:", OUT_PATH, out.shape)
out.head(10)

Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/projected_staffing_by_hour.csv (6680, 10)


,Date,Hour,RxCount,RxCount_Smoothed,RequiredStaff,ScheduledStaff,ScheduledHeadcount,ScheduleSource,OverUnder,OverUnderFlag
0,2023-01-01,9,1,1.500000,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
1,2023-01-01,10,2,1.333333,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
2,2023-01-01,11,1,2.333333,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
3,2023-01-01,12,4,2.333333,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
4,2023-01-01,13,2,2.666667,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
5,2023-01-01,14,2,1.666667,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
6,2023-01-01,15,1,1.666667,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
7,2023-01-01,17,2,1.666667,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
8,2023-01-01,18,2,2.000000,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
9,2023-01-02,8,1,1.000000,2.0,5.774667,6.0,Staff Schedule,3.774667,Overstaffed


In [5]:
# --- 4) Checks ---
print("ScheduleSource distribution:")
print(out["ScheduleSource"].value_counts(), "\n")

print("OverUnderFlag distribution:")
print(out["OverUnderFlag"].value_counts(), "\n")

# Sample day
sample_day = out["Date"].min()
print("Sample day:", sample_day.date())
display(out[out["Date"] == sample_day].sort_values("Hour").head(20))

ScheduleSource distribution:
ScheduleSource
Staff Schedule    6680
Name: count, dtype: int64 

OverUnderFlag distribution:
OverUnderFlag
Overstaffed    6680
Name: count, dtype: int64 

Sample day: 2023-01-01


,Date,Hour,RxCount,RxCount_Smoothed,RequiredStaff,ScheduledStaff,ScheduledHeadcount,ScheduleSource,OverUnder,OverUnderFlag
0,2023-01-01,9,1,1.500000,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
1,2023-01-01,10,2,1.333333,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
2,2023-01-01,11,1,2.333333,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
3,2023-01-01,12,4,2.333333,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
4,2023-01-01,13,2,2.666667,2.0,6.192667,6.0,Staff Schedule,4.192667,Overstaffed
5,2023-01-01,14,2,1.666667,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
6,2023-01-01,15,1,1.666667,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
7,2023-01-01,17,2,1.666667,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
8,2023-01-01,18,2,2.000000,2.0,4.848857,5.0,Staff Schedule,2.848857,Overstaffed
